<a href="https://colab.research.google.com/github/ayachraiet88-crypto/customer-churn/blob/main/Iris_Flower_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
iris=load_iris()

In [ ]:
X=pd.DataFrame(iris.data,columns=iris.feature_names)

In [ ]:
y=iris.target

In [ ]:
print(X.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2


In [ ]:
print(X.shape)

(150, 4)


In [ ]:
print(pd.Series(y).value_counts())

0    50
1    50
2    50
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
from sklearn.linear_model import LassoCV

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)

In [ ]:
lasso=LassoCV(cv=5,random_state=42)
lasso.fit(X_train_scaled,y_train)

LassoCV(cv=5, random_state=42)

In [ ]:
for nom, coef in zip(X.columns,lasso.coef_):
  print(nom,":",round(coef,3))

sepal length (cm) : -0.0
sepal width (cm) : -0.039
petal length (cm) : 0.258
petal width (cm) : 0.512


In [ ]:
from sklearn.neighbors import KNeighborsClassifier


In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
X_test_scaled=scaler.transform(X_test)

In [ ]:
parametres_knn={
    "n_neighbors":[3,5,7,9,11],
    "weights":["uniform","distance"]
}

In [ ]:
grid_knn=GridSearchCV(KNeighborsClassifier(),parametres_knn,cv=5,scoring="accuracy")
grid_knn.fit(X_train_scaled,y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 7, 9, 11],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy')

In [ ]:
print("Meilleur parametres:",grid_knn.best_params_)
print("Meilleur score(validation croisee):",grid_knn.best_score_)

Meilleur parametres: {'n_neighbors': 5, 'weights': 'uniform'}
Meilleur score(validation croisee): 0.9666666666666668


In [ ]:
y_pred=grid_knn.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.83      1.00      0.91        10
           2       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30



In [ ]:
import joblib

In [ ]:
joblib.dump(grid_knn,"iris_model.pkl")

['iris_model.pkl']

In [ ]:
joblib.dump(scaler,"iris_scaler.pkl")

['iris_scaler.pkl']

In [ ]:
pip install gradio

In [ ]:
%%writefile app_iris.py
import gradio as gr
import joblib
import numpy as np

model = joblib.load("iris_model.pkl")
scaler = joblib.load("iris_scaler.pkl")

especes = ["Setosa", "Versicolor", "Virginica"]

def predire_iris(sepal_length, sepal_width, petal_length, petal_width):
    donnees = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    donnees_scaled = scaler.transform(donnees)

    prediction = model.predict(donnees_scaled)[0]
    probabilites = model.predict_proba(donnees_scaled)[0]

    resultat = {especes[i]: float(probabilites[i]) for i in range(3)}
    return resultat

interface = gr.Interface(
    fn=predire_iris,
    inputs=[
        gr.Slider(4.0, 8.0, label="Longueur sépale (cm)"),
        gr.Slider(2.0, 4.5, label="Largeur sépale (cm)"),
        gr.Slider(1.0, 7.0, label="Longueur pétale (cm)"),
        gr.Slider(0.1, 2.5, label="Largeur pétale (cm)")
    ],
    outputs=gr.Label(num_top_classes=3),
    title="🌸 Iris Flower Classifier",
    description="Ajuste les mesures pour prédire l'espèce d'iris, avec les probabilités pour chaque classe."
)

interface.launch()
interface.launch(
    share=True,
    debug=True
)

Overwriting app_iris.py


In [ ]:
!python app_iris.py

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3367, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/app_iris.py", line 26, in <module>
    interface.launch()
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3268, in launch
    self.block_thread()
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3371, in block_thread
    self.server.close()
  File "/usr/local/lib/python3.12/dist-packages/gradio/http_server.py", line 82, in close
    self.thread.join(timeout=5)
  File "/usr/lib/python3.12/threading.py", line 1153, in join
    self._wait_for_tstate_lock(timeout=max(timeout, 0))
  File "/usr/lib/pyth

In [ ]:
%run app_iris.py

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://504df03df464bc3edc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://504df03df464bc3edc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
